# ML 데이터 준비
Purpose: validate the attached shared Dataset and build deterministic row-based ML views.
> Warning: this is an oracle/sanity-only synthetic-data benchmark, not real-device or medical-performance evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path
from typing import Any, Iterable, NamedTuple

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False

ML_OUTPUT_ROOT = Path("/kaggle/working/goal15_ml_view")
DL_TIMELINE_PREFIX = "dl_timeline_"
ORACLE_FEATURE_DENYLIST = (
    "active_target_",
    "hard_negative_id",
    "hard_negative_type",
    "artifact_schedule_id",
    "participant_truth_baseline",
    "event_intensity_truth",
)
STABLE_KEYS = ["person_key", "canonical_time"]
STAGE_JOIN_KEYS = ["run_id", "person_id", "canonical_time"]
EVENT_STAGE_CODES = ("LOW", "MEDIUM", "HIGH", "DECREASING", "RECOVERY")
STAGE_CODES = {"NO_EVENT", *EVENT_STAGE_CODES}
BEHAVIOR_CODES = (
    "ear_covering",
    "exit_attempt",
    "head_turn_away",
    "motion_freeze",
    "movement_reduction",
    "repetitive_body_movement",
    "repetitive_hand_movement",
    "repetitive_object_contact",
    "sustained_pressure_or_contact",
    "withdrawal_movement",
)
CONTEXT_CODES = {
    "sleep",
    "transition",
    "meal_context",
    "focused_task",
    "moderate_activity",
    "light_activity",
    "wake_rest",
    "sedentary_activity",
}
MATCHED_BASELINE_KEYS = ("person_key", "run_id", "context")
DL_CAUSAL_FACTORS = (
    "autonomic_arousal", "motor_activation", "cognitive_load", "sleep_pressure",
    "sensory_context", "recovery_capacity", "social_context",
)
DL_ROLLING_STATISTICS = ("mean", "std", "slope")
DL_ROLLING_WINDOWS_SECONDS = (5, 15, 30, 60, 180, 300)
DL_CAUSAL_FEATURE_COLUMNS = tuple(
    [
        feature
        for factor in DL_CAUSAL_FACTORS
        for feature in (
            f"{factor}__robust_z",
            *(
                f"{factor}__{statistic}_{window_seconds}s"
                for window_seconds in DL_ROLLING_WINDOWS_SECONDS
                for statistic in DL_ROLLING_STATISTICS
            ),
        )
    ]
    + ["time_sin", "time_cos", "weekday_sin", "weekday_cos", "is_awake"]
    + [
        "context__sleep", "context__transition", "context__meal_context",
        "context__focused_task", "context__moderate_activity",
        "context__light_activity", "context__wake_rest",
        "context__sedentary_activity",
    ]
)

class CanonicalPreparedSource(NamedTuple):
    person_key: str
    run_id: str
    person_id: str
    dataset_id: str
    relative_path: str
    declared_path: Path
    canonical_path: Path
    stat_identity: tuple[int, int, int, int]


## 1. Validate the immutable shared Dataset
The notebook reads only flat Dataset files. It never reads hidden truth directories or writes to the attached input.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _schema_fingerprint(schema: pa.Schema) -> str:
    return hashlib.sha256(schema.remove_metadata().serialize().to_pybytes()).hexdigest()


def _physical_split_roles(split_path: Path) -> dict[str, str]:
    parquet = pq.ParquetFile(split_path)
    required = {"person_key", "split_role"}
    if not required.issubset(parquet.schema_arrow.names):
        raise ValueError("split registry missing physical identity columns")
    assignments: dict[str, str] = {}
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group, columns=["person_key", "split_role"]).to_pandas()
        if frame[["person_key", "split_role"]].isna().any().any():
            raise ValueError("split registry has null physical identity")
        for row in frame.itertuples(index=False):
            if not all(isinstance(value, str) and value and value == value.strip() for value in row):
                raise ValueError("split registry has invalid physical identity")
            if row.split_role not in EXPECTED_SPLIT_COUNTS:
                raise ValueError("split registry has invalid split role")
            previous = assignments.setdefault(row.person_key, row.split_role)
            if previous != row.split_role:
                raise ValueError("source person has multiple split roles")
    return assignments


def _physical_person_group(parquet: pq.ParquetFile) -> tuple[str, str, str, int]:
    columns = ["person_key", "run_id", "person_id"]
    if not set(columns).issubset(parquet.schema_arrow.names):
        raise ValueError("prepared source missing physical identity columns")
    counts: dict[tuple[str, str, str], int] = {}
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group, columns=columns).to_pandas()
        if frame[columns].isna().any().any():
            raise ValueError("prepared source has null physical identity")
        for column in columns:
            if not frame[column].map(
                lambda value: isinstance(value, str) and bool(value) and value == value.strip()
            ).all():
                raise ValueError(f"prepared source has invalid physical {column}")
        for key, count in frame.value_counts(columns, sort=False).items():
            counts[tuple(key)] = counts.get(tuple(key), 0) + int(count)
    if len(counts) != 1:
        raise ValueError("prepared source must contain one physical person/run identity")
    (person_key, run_id, person_id), row_count = next(iter(counts.items()))
    if row_count != parquet.metadata.num_rows:
        raise ValueError("prepared source physical row count mismatch")
    return person_key, run_id, person_id, row_count


def _source_stat_identity(path: Path) -> tuple[int, int, int, int]:
    stat = path.stat()
    return (int(stat.st_dev), int(stat.st_ino), int(stat.st_size), int(stat.st_mtime_ns))


def _assert_source_unchanged(source: CanonicalPreparedSource) -> None:
    try:
        resolved_now = source.declared_path.resolve(strict=True)
        stat_now = _source_stat_identity(source.canonical_path)
    except (FileNotFoundError, OSError) as error:
        raise ValueError("prepared physical source changed after preflight") from error
    if resolved_now != source.canonical_path or stat_now != source.stat_identity:
        raise ValueError("prepared physical source changed after preflight")


def _canonical_prepared_sources(
    dataset_root: Path, prepared_entries: list[dict[str, Any]]
) -> tuple[CanonicalPreparedSource, ...]:
    resolved_root = dataset_root.resolve()
    resolved_sources: set[Path] = set()
    sources: list[CanonicalPreparedSource] = []
    for entry in prepared_entries:
        relative_path = f"prepared__people__{entry['dataset_id']}.parquet"
        declared_path = dataset_root / relative_path
        if not declared_path.is_file():
            raise FileNotFoundError(f"missing prepared source file: {entry['dataset_id']}")
        canonical_path = declared_path.resolve(strict=True)
        try:
            canonical_path.relative_to(resolved_root)
        except ValueError as error:
            raise ValueError("prepared physical source path escapes dataset root") from error
        if canonical_path in resolved_sources:
            raise ValueError("duplicate prepared manifest physical source path")
        resolved_sources.add(canonical_path)
        sources.append(CanonicalPreparedSource(
            person_key=entry["person_key"], run_id=entry["run_id"],
            person_id=entry["person_id"], dataset_id=entry["dataset_id"],
            relative_path=relative_path, declared_path=declared_path.absolute(),
            canonical_path=canonical_path, stat_identity=_source_stat_identity(canonical_path),
        ))
    return tuple(sources)


def derive_source_dataset_identity(
    dataset_root: Path,
    manifest_hashes: dict[str, str],
    prepared_entries: list[dict[str, Any]],
) -> tuple[str, list[dict[str, Any]], str]:
    source_hash, inventory, inventory_hash, _ = _preflight_prepared_sources(
        dataset_root, manifest_hashes, {"people": prepared_entries}
    )
    return source_hash, inventory, inventory_hash


def _validate_sha256(value: Any, field: str) -> str:
    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:
        raise ValueError(f"invalid SHA-256 for {field}")
    return value


def _validate_file_hash(dataset_root: Path, filename: str, expected: Any) -> None:
    path = dataset_root / filename
    if not path.is_file():
        raise FileNotFoundError(f"missing dataset file: {filename}")
    expected_hash = _validate_sha256(expected, filename)
    if sha256_file(path) != expected_hash:
        raise ValueError(f"hash mismatch for {filename}")


def resolve_kaggle_dataset_root(input_root: Path = Path("/kaggle/input")) -> Path:
    if not input_root.is_dir():
        raise FileNotFoundError(f"Kaggle attached input root not found: {input_root}")
    manifest_paths = sorted(input_root.rglob("prepared__manifest.json"))
    candidates = sorted({path.parent for path in manifest_paths})
    matches: list[Path] = []
    for candidate in candidates:
        try:
            validate_manifest_hashes(candidate)
        except (FileNotFoundError, ValueError, json.JSONDecodeError):
            continue
        matches.append(candidate)
    if len(matches) != 1:
        raise ValueError(
            f"exactly one hash-validated attached raw Dataset is required: {len(matches)}"
        )
    return matches[0]


def load_split_registry(dataset_root: Path) -> pd.DataFrame:
    split_path = dataset_root / "registry__splits.parquet"
    if not split_path.is_file():
        raise FileNotFoundError(f"missing split registry: {split_path}")
    split = pd.read_parquet(split_path)
    required_columns = {"person_key", "split_role"}
    missing = required_columns.difference(split.columns)
    if missing:
        raise ValueError(f"split registry missing columns: {sorted(missing)}")
    validate_split_contract(split)
    return split[["person_key", "split_role"]].drop_duplicates()


def validate_split_contract(split: pd.DataFrame) -> None:
    counts = split.groupby("split_role")["person_key"].nunique().to_dict()
    if counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(f"split mismatch: {counts}")
    overlaps = [
        set(split.loc[split["split_role"] == role, "person_key"])
        for role in EXPECTED_SPLIT_COUNTS
    ]
    if any(overlaps[i] & overlaps[j] for i in range(3) for j in range(i + 1, 3)):
        raise ValueError("person leakage across split roles")


def validate_manifest_hashes(dataset_root: Path) -> dict[str, str]:
    manifest_names = ("prepared__manifest.json", "outcomes__manifest.json", "registry__manifest.json")
    manifests: dict[str, dict[str, Any]] = {}
    hashes: dict[str, str] = {}
    for name in manifest_names:
        manifest_path = dataset_root / name
        if not manifest_path.is_file():
            raise FileNotFoundError(f"missing manifest: {name}")
        manifest = json.loads(manifest_path.read_text())
        declared_series = manifest.get("series_id") or manifest.get("dataset_id")
        if declared_series is not None and declared_series != SERIES_ID:
            raise ValueError(f"unexpected series in {name}: {declared_series}")
        manifests[name] = manifest
        hashes[name] = sha256_file(manifest_path)

    prepared = manifests["prepared__manifest.json"]
    people = prepared.get("people")
    if not isinstance(people, list) or not people:
        raise ValueError("prepared manifest must declare people")
    for person in people:
        if not isinstance(person, dict) or not isinstance(person.get("dataset_id"), str):
            raise ValueError("prepared manifest has invalid person entry")
        _validate_sha256(person.get("logical_hash"), f"logical hash for {person['dataset_id']}")
        person_path = dataset_root / f"prepared__people__{person['dataset_id']}.parquet"
        if not person_path.is_file():
            raise FileNotFoundError(f"missing dataset file: {person_path.name}")
    for hash_field, filename in (
        ("personal_baseline_sha256", "prepared__personal_baseline.parquet"),
        ("source_split_sha256", "registry__splits.parquet"),
    ):
        if hash_field not in prepared:
            raise ValueError(f"missing required hash: {hash_field}")
        _validate_file_hash(dataset_root, filename, prepared[hash_field])

    outcomes = manifests["outcomes__manifest.json"]
    outcome_files = outcomes.get("files")
    if not isinstance(outcome_files, dict) or not outcome_files:
        raise ValueError("outcomes manifest must declare file hashes")
    for required_name in ("outcome_events.parquet", "outcome_stages.parquet", "outcome_behaviors.parquet"):
        if required_name not in outcome_files:
            raise ValueError(f"missing required outcome hash: {required_name}")
    for relative_name, expected_hash in outcome_files.items():
        if not isinstance(relative_name, str):
            raise ValueError("outcomes manifest has invalid filename")
        _validate_file_hash(dataset_root, f"outcomes__{relative_name}", expected_hash)

    registry = manifests["registry__manifest.json"]
    if "records_sha256" not in registry:
        raise ValueError("missing required hash: records_sha256")
    _validate_file_hash(dataset_root, "registry__registry.jsonl", registry["records_sha256"])
    registry_path = dataset_root / "registry__registry.jsonl"
    if not registry_path.is_file():
        raise FileNotFoundError("missing dataset file: registry__registry.jsonl")
    registry_hashes = {}
    for line in registry_path.read_text().splitlines():
        record = json.loads(line)
        dataset_id = record.get("dataset_id")
        logical_hash = record.get("logical_hash")
        if not isinstance(dataset_id, str):
            raise ValueError("registry record missing dataset_id")
        registry_hashes[dataset_id] = _validate_sha256(logical_hash, f"logical hash for {dataset_id}")
    for person in people:
        if registry_hashes.get(person["dataset_id"]) != person["logical_hash"]:
            raise ValueError(f"logical hash mismatch for {person['dataset_id']}")
    return hashes


def assert_no_truth_leakage(columns: Iterable[str]) -> None:
    leaked = [
        column
        for column in columns
        if any(column == token or column.startswith(token) for token in ORACLE_FEATURE_DENYLIST)
    ]
    if leaked:
        raise ValueError(f"truth leakage columns: {sorted(leaked)}")


def _drop_oracle_columns(frame: pd.DataFrame) -> pd.DataFrame:
    denied = [
        column for column in frame.columns
        if any(column == token or column.startswith(token) for token in ORACLE_FEATURE_DENYLIST)
    ]
    return frame.drop(columns=denied, errors="ignore")


def handoff_schema_hashes() -> tuple[str, str]:
    label_payload = json.dumps(
        {
            "pattern": "pattern_binary",
            "onset_audit": "event_binary",
            "stages": EVENT_STAGE_CODES,
            "behaviors": BEHAVIOR_CODES,
        },
        sort_keys=True,
    )
    feature_payload = json.dumps(
        list(DL_CAUSAL_FEATURE_COLUMNS), separators=(",", ":")
    )
    return (
        hashlib.sha256(label_payload.encode()).hexdigest(),
        hashlib.sha256(feature_payload.encode()).hexdigest(),
    )


## 2. Build deterministic ML row views
Training keeps every positive and hard negative, then takes at most three deterministically ordered baseline rows per positive. Validation and locked test keep their full 1 Hz timelines.

In [ ]:
def _load_outcome_labels(dataset_root: Path, run_id: str | None = None, person_id: str | None = None) -> pd.DataFrame:
    manifest_path = dataset_root / "outcomes__manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("missing manifest: outcomes__manifest.json")
    outcome_files = json.loads(manifest_path.read_text()).get("files")
    if not isinstance(outcome_files, dict):
        raise ValueError("outcomes manifest must declare file hashes")
    for required_name in ("outcome_stages.parquet", "outcome_behaviors.parquet"):
        if required_name not in outcome_files:
            raise ValueError(f"missing required outcome hash: {required_name}")
        _validate_file_hash(dataset_root, f"outcomes__{required_name}", outcome_files[required_name])
    if "outcome_events.parquet" not in outcome_files:
        raise ValueError("missing required outcome hash: outcome_events.parquet")
    _validate_file_hash(dataset_root, "outcomes__outcome_events.parquet", outcome_files["outcome_events.parquet"])

    filters = [("run_id", "==", run_id), ("person_id", "==", person_id)] if run_id is not None and person_id is not None else None
    stages = pd.read_parquet(dataset_root / "outcomes__outcome_stages.parquet", filters=filters).copy()
    required_stage_columns = {"run_id", "person_id", "timestamp_utc", "event_id", "stage_code"}
    missing_stage_columns = required_stage_columns.difference(stages.columns)
    if missing_stage_columns:
        raise ValueError(f"outcome stages missing columns: {sorted(missing_stage_columns)}")
    stages = stages.rename(columns={"timestamp_utc": "canonical_time"})
    stages["canonical_time"] = pd.to_datetime(stages["canonical_time"], utc=True)
    invalid_stages = set(stages["stage_code"].dropna()).difference(STAGE_CODES)
    if invalid_stages:
        raise ValueError(f"invalid stage codes: {sorted(invalid_stages)}")
    if stages["stage_code"].isna().any() or stages.duplicated(STAGE_JOIN_KEYS).any():
        raise ValueError("outcome stages must be complete and unique per run/person/time")

    behaviors = pd.read_parquet(dataset_root / "outcomes__outcome_behaviors.parquet", filters=filters).copy()
    required_behavior_columns = {"run_id", "person_id", "event_id", "behavior_code", "label_value"}
    missing_behavior_columns = required_behavior_columns.difference(behaviors.columns)
    if missing_behavior_columns:
        raise ValueError(f"outcome behaviors missing columns: {sorted(missing_behavior_columns)}")
    if behaviors["behavior_code"].isna().any():
        raise ValueError("behavior_code must be non-null")
    unknown_behaviors = set(behaviors["behavior_code"]).difference(BEHAVIOR_CODES)
    if unknown_behaviors:
        raise ValueError(f"unknown behavior codes: {sorted(unknown_behaviors)}")
    if behaviors["label_value"].isna().any():
        raise ValueError("label_value must be non-null")
    if not behaviors["label_value"].isin((0, 1)).all():
        raise ValueError("label_value must be exactly 0 or 1")
    behavior_keys = ["run_id", "person_id", "event_id", "behavior_code"]
    conflicts = behaviors.groupby(behavior_keys, dropna=False)["label_value"].nunique(dropna=False)
    if (conflicts > 1).any():
        raise ValueError("conflicting behavior labels")
    behavior_matrix = behaviors.pivot_table(
        index=["run_id", "person_id", "event_id"],
        columns="behavior_code",
        values="label_value",
        aggfunc="max",
        fill_value=0,
    ).reindex(columns=BEHAVIOR_CODES, fill_value=0).reset_index()
    labels = stages.merge(behavior_matrix, on=["run_id", "person_id", "event_id"], how="left", validate="many_to_one")
    labels[list(BEHAVIOR_CODES)] = labels[list(BEHAVIOR_CODES)].fillna(0).astype("int8")
    events = pd.read_parquet(dataset_root / "outcomes__outcome_events.parquet", filters=filters)
    event_columns = {"run_id", "person_id", "event_id", "start_time_ns", "end_time_ns"}
    if not event_columns.issubset(events.columns):
        raise ValueError(f"outcome events missing columns: {sorted(event_columns.difference(events.columns))}")
    event_behaviors = events.merge(behaviors[["run_id", "person_id", "event_id", "behavior_code", "label_value"]], on=["run_id", "person_id", "event_id"], how="inner", validate="one_to_many")
    timeline_ns = labels["canonical_time"].map(lambda value: value.value)
    for _, event in event_behaviors.iterrows():
        if pd.isna(event["start_time_ns"]) or pd.isna(event["end_time_ns"]):
            raise ValueError(f"event interval missing bounds: {event['event_id']}")
        mask = (labels["run_id"] == event["run_id"]) & (labels["person_id"] == event["person_id"]) & (timeline_ns >= int(event["start_time_ns"])) & (timeline_ns <= int(event["end_time_ns"]))
        labels.loc[mask, event["behavior_code"]] = labels.loc[mask, event["behavior_code"]].clip(lower=int(event["label_value"]))
    labels[list(BEHAVIOR_CODES)] = labels[list(BEHAVIOR_CODES)].astype("int8")
    return labels


def _validate_matched_baseline_keys(frame: pd.DataFrame) -> None:
    required = {*MATCHED_BASELINE_KEYS, "canonical_time"}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"matched baseline keys missing: {missing}")
    for column in MATCHED_BASELINE_KEYS:
        if frame[column].isna().any() or not frame[column].map(
            lambda value: isinstance(value, str)
            and bool(value)
            and value == value.strip()
        ).all():
            raise ValueError(f"invalid matched baseline {column}")
    if not frame["context"].isin(CONTEXT_CODES).all():
        raise ValueError("invalid matched baseline context")
    if frame["canonical_time"].isna().any():
        raise ValueError("invalid matched baseline canonical_time")


def _deterministic_order(frame: pd.DataFrame) -> pd.DataFrame:
    _validate_matched_baseline_keys(frame)
    ordered = frame.copy()
    identity_columns = [
        column
        for column in (
            "dataset_id",
            "person_key",
            "run_id",
            "person_id",
            "context",
            "canonical_time",
        )
        if column in ordered.columns
    ]
    ordered["_sample_hash"] = pd.util.hash_pandas_object(
        ordered[identity_columns], index=False, categorize=True
    )
    return ordered.sort_values(
        ["_sample_hash", *MATCHED_BASELINE_KEYS, "canonical_time"],
        kind="mergesort",
    )


def bind_source_segment_identity(
    frame: pd.DataFrame,
    *,
    dataset_id: str,
) -> pd.DataFrame:
    if not isinstance(dataset_id, str) or not dataset_id or dataset_id != dataset_id.strip():
        raise ValueError("dataset_id must be a non-empty canonical source id")
    if frame.empty or "canonical_time" not in frame:
        raise ValueError("source frame requires canonical_time rows")
    work = frame.copy()
    canonical_time = pd.to_datetime(work["canonical_time"], errors="raise")
    if not isinstance(canonical_time.dtype, pd.DatetimeTZDtype) or str(canonical_time.dtype.tz) != "UTC":
        raise ValueError("source canonical_time must be UTC-aware")
    work["canonical_time"] = canonical_time
    for column in ("run_id", "person_id"):
        if column not in work or work[column].isna().any() or not work[column].map(
            lambda value: isinstance(value, str) and bool(value) and value == value.strip()
        ).all():
            raise ValueError(f"source identity is invalid: {column}")
    if "dataset_id" in work and not work["dataset_id"].eq(dataset_id).all():
        raise ValueError("source dataset_id conflicts with physical manifest")
    work["dataset_id"] = dataset_id
    expected_day = canonical_time.dt.strftime("%Y-%m-%d")
    if "day_key" in work and not work["day_key"].eq(expected_day).all():
        raise ValueError("source day_key conflicts with canonical UTC day")
    work["day_key"] = expected_day
    if "session_id" in work:
        invalid_session = work["session_id"].isna() | ~work["session_id"].map(
            lambda value: isinstance(value, str) and bool(value) and value == value.strip()
        )
        if invalid_session.any():
            raise ValueError("source session_id is invalid")
    else:
        work["session_id"] = dataset_id
    return work


def build_ml_role_view(
    prepared: pd.DataFrame,
    labels: pd.DataFrame,
    split: pd.DataFrame,
    split_role: str,
) -> pd.DataFrame:
    if split_role not in EXPECTED_SPLIT_COUNTS:
        raise ValueError(f"unknown split role: {split_role}")
    assert_no_truth_leakage(prepared.columns)
    _validate_matched_baseline_keys(prepared)
    role_people = split.loc[split["split_role"] == split_role, ["person_key"]]
    feature_rows = prepared.merge(role_people, on="person_key", how="inner", validate="many_to_one")
    if not set(STAGE_JOIN_KEYS).issubset(feature_rows.columns) or not set(STAGE_JOIN_KEYS).issubset(labels.columns):
        raise ValueError("ML label joins require run_id, person_id, and canonical_time")
    label_join_keys = STAGE_JOIN_KEYS
    view = feature_rows.merge(labels, on=label_join_keys, how="left", validate="one_to_one")
    label_columns = [column for column in labels.columns if column not in label_join_keys]
    view[label_columns] = view[label_columns].fillna(0)
    required_multitask_columns = {"stage_code", *BEHAVIOR_CODES}
    missing_multitask_columns = required_multitask_columns.difference(view.columns)
    if missing_multitask_columns:
        raise ValueError(f"missing multitask labels: {sorted(missing_multitask_columns)}")
    invalid_stages = set(view["stage_code"].dropna()).difference(STAGE_CODES)
    if invalid_stages or (view["stage_code"] == 0).any():
        raise ValueError("missing or invalid stage_code")
    view["pattern_binary"] = (view["stage_code"] != "NO_EVENT").astype("int8")
    view[list(BEHAVIOR_CODES)] = view[list(BEHAVIOR_CODES)].fillna(0).astype("int8")
    event_columns = [column for column in ("pattern_binary", "event_label", "event_binary", "label") if column in view]
    hard_negative_columns = [
        column for column in ("hard_negative", "is_hard_negative", "hard_negative_id", "hard_negative_type")
        if column in view
    ]
    if not event_columns:
        raise ValueError("outcome events need event_label, event_binary, or label")
    positive_mask = view["pattern_binary"].eq(1)
    hard_negative_mask = pd.Series(False, index=view.index)
    for column in hard_negative_columns:
        values = view[column]
        if column in {"hard_negative_id", "hard_negative_type"}:
            hard_negative_mask |= values.notna() & values.ne(0) & values.astype(str).str.strip().ne("")
        else:
            hard_negative_mask |= values.fillna(False).astype(bool)
    if split_role != "train":
        sanitized = _drop_oracle_columns(view).drop(columns=["event_id"], errors="ignore")
        assert_no_truth_leakage(sanitized.columns)
        return sanitized.sort_values(STABLE_KEYS, kind="mergesort").reset_index(drop=True)
    required_rows = view.loc[positive_mask | hard_negative_mask]
    baseline_candidates = view.loc[~(positive_mask | hard_negative_mask)]
    positive_counts = (
        view.loc[positive_mask]
        .groupby(list(MATCHED_BASELINE_KEYS), sort=False, dropna=False)
        .size()
    )
    sampled_groups: list[pd.DataFrame] = []
    for stratum, positive_count in positive_counts.items():
        stratum_values = stratum if isinstance(stratum, tuple) else (stratum,)
        in_stratum = pd.Series(True, index=baseline_candidates.index)
        for column, value in zip(MATCHED_BASELINE_KEYS, stratum_values, strict=True):
            in_stratum &= baseline_candidates[column].eq(value)
        ordered = _deterministic_order(baseline_candidates.loc[in_stratum])
        sampled_groups.append(ordered.head(3 * int(positive_count)))
    sampled_baselines = (
        pd.concat(sampled_groups, ignore_index=False)
        if sampled_groups
        else baseline_candidates.iloc[0:0].copy()
    )
    sampled = pd.concat([required_rows, sampled_baselines], ignore_index=True)
    internal_columns = [column for column in sampled if column.startswith("_sample_")]
    sampled = sampled.drop(columns=internal_columns, errors="ignore")
    sampled = sampled.drop(columns=["event_id"], errors="ignore")
    sampled = _drop_oracle_columns(sampled)
    assert_no_truth_leakage(sampled.columns)
    return sampled.sort_values(STABLE_KEYS, kind="mergesort").reset_index(drop=True)


def build_full_dl_role_view(
    prepared: pd.DataFrame,
    labels: pd.DataFrame,
    split: pd.DataFrame,
    split_role: str,
    *,
    dataset_id: str,
) -> pd.DataFrame:
    if split_role not in EXPECTED_SPLIT_COUNTS:
        raise ValueError(f"unknown split role: {split_role}")
    if not isinstance(dataset_id, str) or not dataset_id or dataset_id != dataset_id.strip():
        raise ValueError("full DL timeline requires a canonical dataset_id")
    assert_no_truth_leakage(prepared.columns)
    role_people = split.loc[split["split_role"] == split_role, ["person_key"]]
    feature_rows = prepared.merge(role_people, on="person_key", how="inner", validate="many_to_one")
    if not set(STAGE_JOIN_KEYS).issubset(feature_rows.columns) or not set(STAGE_JOIN_KEYS).issubset(labels.columns):
        raise ValueError("DL label joins require run_id, person_id, and canonical_time")
    label_columns = [
        column for column in labels.columns
        if column not in STAGE_JOIN_KEYS and column not in feature_rows.columns
    ]
    view = feature_rows.merge(
        labels[[*STAGE_JOIN_KEYS, *label_columns]],
        on=STAGE_JOIN_KEYS,
        how="left",
        validate="one_to_one",
    )
    view[label_columns] = view[label_columns].fillna(0)
    required_labels = {"stage_code", *BEHAVIOR_CODES}
    missing_labels = required_labels.difference(view.columns)
    if missing_labels:
        raise ValueError(f"missing multitask labels: {sorted(missing_labels)}")
    if view["stage_code"].isna().any() or not view["stage_code"].isin(STAGE_CODES).all():
        raise ValueError("missing or invalid stage_code")
    view["pattern_binary"] = view["stage_code"].ne("NO_EVENT").astype("int8")
    view[list(BEHAVIOR_CODES)] = view[list(BEHAVIOR_CODES)].fillna(0).astype("int8")
    hard_negative = pd.Series(False, index=view.index)
    for column in ("hard_negative", "is_hard_negative", "hard_negative_id", "hard_negative_type"):
        if column not in view:
            continue
        values = view[column]
        if column in {"hard_negative_id", "hard_negative_type"}:
            hard_negative |= values.notna() & values.ne(0) & values.astype(str).str.strip().ne("")
        else:
            hard_negative |= values.fillna(False).astype(bool)
    view["hard_negative"] = hard_negative.astype("int8")
    view["dataset_id"] = dataset_id
    view["split_role"] = split_role
    sanitized = _drop_oracle_columns(view).drop(columns=["event_id"], errors="ignore")
    assert_no_truth_leakage(sanitized.columns)
    return sanitized.sort_values(STABLE_KEYS, kind="mergesort").reset_index(drop=True)


def write_ml_view_manifest(
    output_root: Path,
    view_paths: dict[str, Path],
    source_dataset_hash: str,
    split_hash: str,
    row_counts: dict[str, int],
    role_columns: dict[str, list[str]],
    dl_timeline_paths: dict[str, Path] | None = None,
    dl_timeline_row_counts: dict[str, int] | None = None,
    dl_timeline_columns: dict[str, list[str]] | None = None,
    dl_timeline_dataset_ids: dict[str, set[str]] | None = None,
    source_content_inventory: list[dict[str, Any]] | None = None,
    source_content_inventory_hash: str | None = None,
) -> Path:
    roles = set(view_paths)
    if set(row_counts) != roles or set(role_columns) != roles:
        raise ValueError("manifest metadata roles do not match view outputs")
    files: dict[str, dict[str, Any]] = {}
    for split_role, view_path in view_paths.items():
        row_count = row_counts[split_role]
        columns = role_columns[split_role]
        if isinstance(row_count, bool) or not isinstance(row_count, int) or row_count < 0:
            raise ValueError(f"invalid row count for {split_role}")
        if not columns or not all(isinstance(column, str) for column in columns):
            raise ValueError(f"invalid columns for {split_role}")
        files[split_role] = {
            "path": view_path.name,
            "sha256": sha256_file(view_path),
            "row_count": row_count,
            "columns": columns,
            "view_kind": "sampled_ml_rows",
            "sampled": split_role == "train",
        }
    dl_timeline_files: dict[str, dict[str, Any]] = {}
    optional_groups = (dl_timeline_paths, dl_timeline_row_counts, dl_timeline_columns, dl_timeline_dataset_ids)
    if any(group is not None for group in optional_groups):
        if any(group is None for group in optional_groups):
            raise ValueError("incomplete full DL timeline metadata")
        assert dl_timeline_paths is not None
        assert dl_timeline_row_counts is not None
        assert dl_timeline_columns is not None
        assert dl_timeline_dataset_ids is not None
        if not all(set(group) == roles for group in optional_groups):
            raise ValueError("full DL timeline roles do not match ML roles")
        for split_role, timeline_path in dl_timeline_paths.items():
            columns = dl_timeline_columns[split_role]
            missing_features = sorted(set(DL_CAUSAL_FEATURE_COLUMNS).difference(columns))
            dataset_ids = sorted(dl_timeline_dataset_ids[split_role])
            if missing_features:
                raise ValueError(f"full DL timeline missing causal features: {missing_features}")
            if not dataset_ids:
                raise ValueError(f"full DL timeline has no dataset ids: {split_role}")
            dl_timeline_files[split_role] = {
                "path": timeline_path.name,
                "sha256": sha256_file(timeline_path),
                "row_count": dl_timeline_row_counts[split_role],
                "columns": columns,
                "feature_columns": list(DL_CAUSAL_FEATURE_COLUMNS),
                "dataset_ids": dataset_ids,
                "view_kind": "full_causal_timeline",
                "sampled": False,
            }
            if pq.ParquetFile(timeline_path).metadata.num_rows != dl_timeline_row_counts[split_role]:
                raise ValueError(f"full DL timeline physical row count mismatch: {split_role}")
    if dl_timeline_files and not source_content_inventory:
        raise ValueError("source content inventory is required")
    if not source_content_inventory:
        source_content_inventory = []
        source_content_inventory_hash = hashlib.sha256(b"[]").hexdigest()
    inventory_hash = _validate_sha256(source_content_inventory_hash, "source content inventory")
    calculated_inventory_hash = hashlib.sha256(
        json.dumps(source_content_inventory, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    if inventory_hash != calculated_inventory_hash:
        raise ValueError("source content inventory hash mismatch")
    for split_role, metadata in dl_timeline_files.items():
        source_groups = sorted(
            [
                {
                    "person_key": item["person_key"],
                    "run_id": item["run_id"],
                    "dataset_id": item["dataset_id"],
                    "split_role": item["split_role"],
                    "row_count": item["row_count"],
                }
                for item in source_content_inventory
                if item.get("split_role") == split_role
            ],
            key=lambda item: (item["person_key"], item["run_id"], item["dataset_id"]),
        )
        if sum(item["row_count"] for item in source_groups) != metadata["row_count"]:
            raise ValueError(f"full DL timeline source group row count mismatch: {split_role}")
        if {item["dataset_id"] for item in source_groups} != set(metadata["dataset_ids"]):
            raise ValueError(f"full DL timeline source group dataset mismatch: {split_role}")
        metadata["source_groups"] = source_groups
    label_schema_hash, feature_schema_hash = handoff_schema_hashes()
    manifest_path = output_root / "view_manifest.json"
    manifest_path.write_text(json.dumps({
        "series_id": SERIES_ID,
        "data_status": DATA_STATUS,
        "source_dataset_hash": source_dataset_hash,
        "split_hash": split_hash,
        "label_schema_hash": label_schema_hash,
        "feature_schema_hash": feature_schema_hash,
        "files": files,
        "dl_timeline_files": dl_timeline_files,
        "source_content_inventory": source_content_inventory,
        "source_content_inventory_hash": inventory_hash,
    }, indent=2, sort_keys=True) + "\n")
    return manifest_path


def ordered_prepared_entries(
    prepared_manifest: dict[str, Any],
    *,
    dataset_root: Path | None = None,
    split_roles: dict[str, str] | None = None,
) -> list[dict[str, Any]]:
    people = prepared_manifest.get("people")
    if not isinstance(people, list) or not people:
        raise ValueError("prepared manifest must declare people")
    required = ("person_key", "run_id", "person_id", "dataset_id")
    for entry in people:
        if not isinstance(entry, dict) or any(
            not isinstance(entry.get(field), str)
            or not entry[field]
            or entry[field] != entry[field].strip()
            for field in required
        ):
            raise ValueError("prepared manifest has invalid sequence identity")
    identities = [tuple(entry[field] for field in required) for entry in people]
    if len(set(identities)) != len(identities):
        raise ValueError("duplicate prepared manifest entry tuple")
    for field in ("person_key", "dataset_id"):
        values = [entry[field] for entry in people]
        if len(set(values)) != len(values):
            raise ValueError(f"duplicate prepared manifest {field}")
    expected_total = sum(EXPECTED_SPLIT_COUNTS.values())
    if len(people) != expected_total:
        raise ValueError(f"prepared manifest must declare exactly {expected_total} entries")
    if split_roles is not None:
        if not isinstance(split_roles, dict) or any(
            not isinstance(person_key, str)
            or not isinstance(role, str)
            or role not in EXPECTED_SPLIT_COUNTS
            for person_key, role in split_roles.items()
        ):
            raise ValueError("split registry has invalid prepared membership")
        role_counts = {
            role: sum(value == role for value in split_roles.values())
            for role in EXPECTED_SPLIT_COUNTS
        }
        if role_counts != EXPECTED_SPLIT_COUNTS:
            raise ValueError(f"split registry role membership mismatch: {role_counts}")
        if {entry["person_key"] for entry in people} != set(split_roles):
            raise ValueError("prepared manifest people must exactly match split registry")
    ordered = sorted(
        people,
        key=lambda entry: (
            entry["person_key"], entry["run_id"], entry["person_id"], entry["dataset_id"]
        ),
    )
    if dataset_root is not None:
        _canonical_prepared_sources(dataset_root, ordered)
    return ordered


def _preflight_prepared_sources(
    dataset_root: Path,
    manifest_hashes: dict[str, str],
    prepared_manifest: dict[str, Any] | None = None,
) -> tuple[str, list[dict[str, Any]], str, tuple[CanonicalPreparedSource, ...]]:
    split_path = dataset_root / "registry__splits.parquet"
    if not split_path.is_file():
        raise FileNotFoundError("missing split registry for source identity")
    if prepared_manifest is None:
        prepared_path = dataset_root / "prepared__manifest.json"
        prepared_bytes = prepared_path.read_bytes()
        prepared_hash = hashlib.sha256(prepared_bytes).hexdigest()
        if prepared_hash != manifest_hashes.get("prepared__manifest.json"):
            raise ValueError("prepared manifest changed before source preflight")
        prepared_manifest = json.loads(prepared_bytes)
    split_roles = _physical_split_roles(split_path)
    prepared_entries = ordered_prepared_entries(
        prepared_manifest, split_roles=split_roles
    )
    prepared_sources = _canonical_prepared_sources(dataset_root, prepared_entries)
    inventory: list[dict[str, Any]] = []
    for source in sorted(prepared_sources, key=lambda item: item.dataset_id):
        _assert_source_unchanged(source)
        parquet = pq.ParquetFile(source.canonical_path)
        person_key, run_id, person_id, row_count = _physical_person_group(parquet)
        if (person_key, run_id, person_id) != (
            source.person_key, source.run_id, source.person_id
        ):
            raise ValueError(f"prepared manifest differs from physical identity: {source.dataset_id}")
        source_sha256 = sha256_file(source.canonical_path)
        schema_fingerprint = _schema_fingerprint(parquet.schema_arrow)
        _assert_source_unchanged(source)
        split_role = split_roles[person_key]
        inventory.append({
            "dataset_id": source.dataset_id,
            "person_key": person_key,
            "run_id": run_id,
            "person_id": person_id,
            "split_role": split_role,
            "path": source.relative_path,
            "sha256": source_sha256,
            "row_count": row_count,
            "schema_fingerprint": schema_fingerprint,
        })
    inventory_payload = json.dumps(inventory, sort_keys=True, separators=(",", ":"))
    inventory_hash = hashlib.sha256(inventory_payload.encode()).hexdigest()
    envelope = {
        "manifest_hashes": dict(sorted(manifest_hashes.items())),
        "prepared_source_inventory_hash": inventory_hash,
        "split_sha256": sha256_file(split_path),
    }
    source_hash = hashlib.sha256(
        json.dumps(envelope, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    return source_hash, inventory, inventory_hash, prepared_sources


def build_all_ml_views(
    *,
    flat_dataset_root: Path | None = None,
    output_root: Path = ML_OUTPUT_ROOT,
) -> Path:
    dataset_root = flat_dataset_root or resolve_kaggle_dataset_root()
    manifest_hashes = validate_manifest_hashes(dataset_root)
    split = load_split_registry(dataset_root)
    source_dataset_hash, source_inventory, source_inventory_hash, prepared_sources = _preflight_prepared_sources(
        dataset_root, manifest_hashes
    )
    person_roles = split.set_index("person_key")["split_role"].to_dict()
    expected_timeline_rows = {role: 0 for role in EXPECTED_SPLIT_COUNTS}
    expected_timeline_dataset_ids = {role: set() for role in EXPECTED_SPLIT_COUNTS}
    for item in source_inventory:
        role = item["split_role"]
        if role not in EXPECTED_SPLIT_COUNTS or person_roles.get(item["person_key"]) != role:
            raise ValueError(f"prepared source is outside immutable split: {item['person_key']}")
        expected_timeline_rows[role] += item["row_count"]
        expected_timeline_dataset_ids[role].add(item["dataset_id"])
    output_root.mkdir(parents=True, exist_ok=True)
    view_paths: dict[str, Path] = {}
    writers: dict[str, pq.ParquetWriter] = {}
    row_counts = {role: 0 for role in EXPECTED_SPLIT_COUNTS}
    role_schemas: dict[str, pa.Schema] = {}
    role_columns: dict[str, list[str]] = {}
    timeline_paths: dict[str, Path] = {}
    timeline_writers: dict[str, pq.ParquetWriter] = {}
    timeline_row_counts = {role: 0 for role in EXPECTED_SPLIT_COUNTS}
    timeline_schemas: dict[str, pa.Schema] = {}
    timeline_columns: dict[str, list[str]] = {}
    timeline_dataset_ids = {role: set() for role in EXPECTED_SPLIT_COUNTS}
    try:
        for source in prepared_sources:
            dataset_id = source.dataset_id
            prepared_path = source.canonical_path
            _assert_source_unchanged(source)
            prepared = pd.read_parquet(prepared_path)
            _assert_source_unchanged(source)
            if "canonical_time" not in prepared and "timestamp_utc" in prepared:
                prepared = prepared.rename(columns={"timestamp_utc": "canonical_time"})
            prepared["canonical_time"] = pd.to_datetime(prepared["canonical_time"], utc=True)
            prepared = bind_source_segment_identity(prepared, dataset_id=dataset_id)
            if prepared["person_key"].nunique() != 1 or prepared["run_id"].nunique() != 1 or prepared["person_id"].nunique() != 1:
                raise ValueError(f"prepared person file is not single-person: {prepared_path.name}")
            for field in ("person_key", "run_id", "person_id"):
                if str(prepared[field].iloc[0]) != getattr(source, field):
                    raise ValueError(f"prepared manifest identity mismatch: {dataset_id}/{field}")
            person_key = str(prepared["person_key"].iloc[0])
            roles = split.loc[split["person_key"] == person_key, "split_role"].unique()
            if len(roles) != 1:
                raise ValueError(f"prepared person has invalid split assignment: {person_key}")
            split_role = str(roles[0])
            labels = _load_outcome_labels(dataset_root, str(prepared["run_id"].iloc[0]), str(prepared["person_id"].iloc[0]))
            view = build_ml_role_view(prepared, labels, split, split_role)
            view_path = output_root / f"{split_role}.parquet"
            table = pa.Table.from_pandas(view, preserve_index=False)
            if split_role not in writers:
                writers[split_role] = pq.ParquetWriter(view_path, table.schema, compression="zstd")
                view_paths[split_role] = view_path
                role_schemas[split_role] = table.schema
                role_columns[split_role] = list(table.schema.names)
            elif not table.schema.equals(role_schemas[split_role], check_metadata=False):
                raise ValueError(f"inconsistent ML view schema for {split_role}")
            writers[split_role].write_table(table)
            row_counts[split_role] += len(view)
            del prepared
            _assert_source_unchanged(source)
            prepared_file = pq.ParquetFile(prepared_path)
            for row_group in range(prepared_file.num_row_groups):
                prepared_chunk = prepared_file.read_row_group(row_group).to_pandas()
                if "canonical_time" not in prepared_chunk and "timestamp_utc" in prepared_chunk:
                    prepared_chunk = prepared_chunk.rename(columns={"timestamp_utc": "canonical_time"})
                prepared_chunk["canonical_time"] = pd.to_datetime(prepared_chunk["canonical_time"], utc=True)
                prepared_chunk = bind_source_segment_identity(prepared_chunk, dataset_id=dataset_id)
                timeline = build_full_dl_role_view(
                    prepared_chunk, labels, split, split_role, dataset_id=dataset_id
                )
                missing_features = sorted(set(DL_CAUSAL_FEATURE_COLUMNS).difference(timeline.columns))
                if missing_features:
                    raise ValueError(f"full DL timeline missing causal features: {missing_features}")
                timeline_path = output_root / f"{DL_TIMELINE_PREFIX}{split_role}.parquet"
                timeline_table = pa.Table.from_pandas(timeline, preserve_index=False)
                if split_role not in timeline_writers:
                    timeline_writers[split_role] = pq.ParquetWriter(
                        timeline_path, timeline_table.schema, compression="zstd"
                    )
                    timeline_paths[split_role] = timeline_path
                    timeline_schemas[split_role] = timeline_table.schema
                    timeline_columns[split_role] = list(timeline_table.schema.names)
                elif not timeline_table.schema.equals(timeline_schemas[split_role], check_metadata=False):
                    raise ValueError(f"inconsistent full DL timeline schema for {split_role}")
                timeline_writers[split_role].write_table(timeline_table)
                timeline_row_counts[split_role] += len(timeline)
                timeline_dataset_ids[split_role].add(dataset_id)
            _assert_source_unchanged(source)
    finally:
        for writer in writers.values():
            writer.close()
        for writer in timeline_writers.values():
            writer.close()
    if set(view_paths) != set(EXPECTED_SPLIT_COUNTS) or set(timeline_paths) != set(EXPECTED_SPLIT_COUNTS):
        raise ValueError("missing ML or full DL timeline split output")
    if timeline_row_counts != expected_timeline_rows:
        raise ValueError("full DL timeline does not preserve every source row")
    if timeline_dataset_ids != expected_timeline_dataset_ids:
        raise ValueError("full DL timeline source dataset binding mismatch")
    for source in prepared_sources:
        _assert_source_unchanged(source)
    split_hash = sha256_file(dataset_root / "registry__splits.parquet")
    return write_ml_view_manifest(
        output_root, view_paths, source_dataset_hash, split_hash, row_counts, role_columns,
        timeline_paths, timeline_row_counts, timeline_columns, timeline_dataset_ids,
        source_inventory, source_inventory_hash,
    )


## 3. Explicit execution gate
Data preparation remains disabled in the committed notebook.

### Kaggle 노트북 인계 순서
`01 → 03 → 02 → 04` 순서로 사용합니다. 이 01 노트북에서 명시적으로 데이터 준비를 실행한 뒤 **Save Version**으로 출력만 저장하고, 그 저장된 출력을 03과 02 노트북에 각각 입력 데이터로 연결합니다. 04에는 01·03·02의 저장된 출력을 모두 입력으로 연결합니다. 커밋된 기본값에서는 실행과 학습이 비활성화되어 있습니다.

In [ ]:
RUN_DATA_PREPARATION = False
if RUN_DATA_PREPARATION:
    build_all_ml_views()
else:
    print("준비 완료: RUN_DATA_PREPARATION=True로 바꿀 때만 데이터를 생성합니다.")